# Adaptive Step 1 planner evaluation

This notebook evaluates the learned schedule—not a fixed-schedule model. Fixed gaps are rate controls used to determine whether adaptation contributes anything.

The protocol separates:

1. final-phase teacher-forced oracle likelihood;
2. adaptive gaps with GT anchor history (placement-only diagnostic);
3. adaptive gaps with generated anchor history (deployable Step 1);
4. fixed-gap and frozen-Step-2-DP controls;
5. anchor-substitution FID (anchor damage only);
6. frozen-Step-2 infilling FID (actual system quality).

The rollout is **offline known-duration**: validation motion length is used only to stop at the exact final frame. The evaluator reports every final gap decision that had to be clipped. Do not describe this notebook as strict unknown-duration online inference.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Robustly locate the repository whether Jupyter starts at the repo root,
# motion_generation/, or motion_generation/notebooks/.
candidate = Path.cwd().resolve()
PROJECT_ROOT = None
for path in (candidate, *candidate.parents):
    if (path / 'motion_generation').is_dir() and (path / 'checkpoints').is_dir():
        PROJECT_ROOT = path
        break
assert PROJECT_ROOT is not None, f'Could not locate project root from {candidate}'

PYTHON_EXECUTABLE = Path(sys.executable).resolve()
DEVICE = 'cuda:0'
PRIMARY_LABEL = 'adaptive_best'
CHECKPOINTS = {
    PRIMARY_LABEL: PROJECT_ROOT / 'checkpoints/step1_multipart_adaptive_gap_step2_curriculum50/best',
    # Add this only when you intentionally want a second complete rollout:
    # 'adaptive_final': PROJECT_ROOT / 'checkpoints/step1_multipart_adaptive_gap_step2_curriculum50/final',
}

ADAPTIVE_OUTPUT = PROJECT_ROOT / 'motion_generation/outputs/step1_adaptive_gap_evaluation'
MOTION_OUTPUT = PROJECT_ROOT / 'motion_generation/outputs/step1_adaptive_motion_evaluation'

# Quick mode validates the protocol on the same deterministic 128 clips.
# Set False for the final 635-clip validation report.
QUICK_MODE = True
TEACHER_MAX_CLIPS = 0
ROLLOUT_MAX_CLIPS = 128 if QUICK_MODE else 0
TEACHER_BATCH_SIZE = 32
ROLLOUT_BATCH_SIZE = 8
STEP2_BATCH_SIZE = 256
SUBSET_SEED = 42

FIXED_GAPS = '3,5,7,9,11,15'
GENERATED_FIXED_GAPS = '7'
RUN_ADAPTIVE_EVALUATION = True
RUN_MOTION_EXPORT = True
RUN_FID = True

print('project:', PROJECT_ROOT)
print('python:', PYTHON_EXECUTABLE)
print('rollout clips:', 'all 635' if ROLLOUT_MAX_CLIPS == 0 else ROLLOUT_MAX_CLIPS)


## Preflight

The adaptive checkpoint requires the final calibration file, its schedule NPZ, and the frozen Step 2 interval-cost cache. The motion evaluator also verifies that the Step 2 weight fingerprint is identical to the checkpoint that created those costs.

In [ ]:
for label, checkpoint in CHECKPOINTS.items():
    assert checkpoint.is_dir(), (label, checkpoint)
    assert (checkpoint / 'phase1_source_config.json').is_file(), checkpoint

CALIBRATION = PROJECT_ROOT / 'checkpoints/step1_adaptive_gap_oracle/calibration.json'
STEP2_CONFIG = PROJECT_ROOT / 'motion_generation/configs/audio_c2f_body_causal_moss_nano_all16_soft_recovery_sf05_stage2.yaml'
STEP2_CHECKPOINT = PROJECT_ROOT / 'checkpoints/mask_multipart_body_causal_moss_nano_all16_variable_c2f_soft_recovery_sf05_stage2_gap1_15'
assert CALIBRATION.is_file(), CALIBRATION
assert STEP2_CONFIG.is_file(), STEP2_CONFIG
assert STEP2_CHECKPOINT.is_dir(), STEP2_CHECKPOINT

calibration = json.loads(CALIBRATION.read_text(encoding='utf-8'))
cost_dir = Path(calibration['cost_dir'])
assert cost_dir.is_dir(), cost_dir
final_phase = max(calibration['phases'], key=lambda row: int(row['phase_index']))
display(pd.DataFrame([{
    'cost_dir': str(cost_dir),
    'step2_checkpoint_used_for_costs': calibration['cost_manifests'][0]['checkpoint'],
    'final_gap_range': f"{final_phase['min_gap']}–{final_phase['max_gap']}",
    'target_mean_gap': final_phase['target_mean_gap'],
    'materialized_mean_gap': final_phase['materialized_mean_gap'],
    'anchor_penalty': final_phase['calibrated_anchor_penalty'],
}]))


## 1. Token- and schedule-level evaluation

The main comparisons are:

- `adaptive_gt_history`: learned locations with GT previous anchors. This isolates schedule quality.
- `adaptive_generated_history`: learned locations and generated anchors. This is deployable Step 1.
- `fixed_gap7_*`: the principal rate-matched control because the final curriculum targeted mean gap 7.
- `step2_dp_oracle_gt_anchors`: the frozen-Step-2 placement upper bound.

In [ ]:
command = [
    str(PYTHON_EXECUTABLE),
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_adaptive_gap.py'),
    '--output_dir', str(ADAPTIVE_OUTPUT),
    '--device', DEVICE,
    '--teacher_max_clips', str(TEACHER_MAX_CLIPS),
    '--rollout_max_clips', str(ROLLOUT_MAX_CLIPS),
    '--teacher_batch_size', str(TEACHER_BATCH_SIZE),
    '--rollout_batch_size', str(ROLLOUT_BATCH_SIZE),
    '--num_workers', '0',
    '--subset_seed', str(SUBSET_SEED),
    '--fixed_gaps', FIXED_GAPS,
    '--generated_fixed_gaps', GENERATED_FIXED_GAPS,
]
for label, checkpoint in CHECKPOINTS.items():
    command.extend(['--checkpoint', f'{label}={checkpoint}'])
print(' '.join(command))
if RUN_ADAPTIVE_EVALUATION:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Skipped; reading existing outputs.')


In [ ]:
teacher = pd.read_csv(ADAPTIVE_OUTPUT / 'teacher_forced_final_phase.csv')
schedule = pd.read_csv(ADAPTIVE_OUTPUT / 'schedule_summary.csv')
gap_histogram = pd.read_csv(ADAPTIVE_OUTPUT / 'gap_histogram.csv')
anchors = pd.read_csv(ADAPTIVE_OUTPUT / 'generated_anchor_summary.csv')

display(teacher)
display(schedule.sort_values('objective_per_frame'))
display(anchors.sort_values('accuracy', ascending=False))


In [ ]:
# Learned/fixed quality-rate frontier. Lower frozen-Step-2 risk is better;
# fewer anchors are cheaper. The oracle is an upper bound, not a deployable result.
plot_frame = schedule.copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for _, row in plot_frame.iterrows():
    axes[0].scatter(row['mean_anchors_including_seed'], row['cached_gt_boundary_step2_risk_per_frame'], s=55)
    axes[0].annotate(row['condition'], (row['mean_anchors_including_seed'], row['cached_gt_boundary_step2_risk_per_frame']), fontsize=8)
axes[0].set_xlabel('Mean anchors per clip (lower is cheaper)')
axes[0].set_ylabel('Cached GT-boundary Step-2 risk per frame (lower is better)')
axes[0].set_title('Schedule quality–rate frontier')
axes[0].grid(alpha=0.25)

adaptive_condition = f'{PRIMARY_LABEL}__adaptive_generated_history'
hist = gap_histogram[(gap_histogram['condition'] == adaptive_condition) & (gap_histogram['kind'] == 'executed')]
axes[1].bar(hist['gap'], hist['fraction'])
axes[1].set_xticks(range(16))
axes[1].set_xlabel('Executed gap')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Adaptive generated-history gap distribution')
axes[1].grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# Fast diagnostic gates before spending time on decoded motion/FID.
def one(condition):
    rows = schedule[schedule['condition'] == condition]
    assert len(rows) == 1, condition
    return rows.iloc[0]

adaptive_gt = one(f'{PRIMARY_LABEL}__adaptive_gt_history')
adaptive_generated = one(f'{PRIMARY_LABEL}__adaptive_generated_history')
fixed7_gt = one('fixed_gap7_gt_anchors')
fixed7_generated = one(f'{PRIMARY_LABEL}__fixed_gap7_generated_history')

diagnostics = pd.DataFrame([
    {
        'gate': 'placement_vs_fixed7',
        'value': adaptive_gt['cached_gt_boundary_step2_risk_per_frame'] - fixed7_gt['cached_gt_boundary_step2_risk_per_frame'],
        'pass_rule': '< 0 at comparable anchor rate',
    },
    {
        'gate': 'generated_history_schedule_penalty',
        'value': adaptive_generated['cached_gt_boundary_step2_risk_per_frame'] - adaptive_gt['cached_gt_boundary_step2_risk_per_frame'],
        'pass_rule': 'small; large positive means exposure bias changes placement',
    },
    {
        'gate': 'adaptive_oracle_regret_per_frame',
        'value': adaptive_gt['oracle_regret_per_frame'],
        'pass_rule': 'closer to 0 is better',
    },
    {
        'gate': 'eos_clipping_fraction',
        'value': adaptive_generated['eos_clipped_fraction'],
        'pass_rule': 'report explicitly; this is offline known-duration control',
    },
])
display(diagnostics)

endpoint_mass = hist.loc[hist['gap'].isin([3, 15]), 'fraction'].sum()
print(f'gap-3/gap-15 endpoint mass: {endpoint_mass:.2%}')
if endpoint_mass > 0.80:
    print('WARNING: the learned scheduler may have collapsed toward curriculum endpoints.')


## 2. Motion export through the exact frozen Step 2

This stage verifies the selected Step 2 checkpoint by weight fingerprint. It exports both anchor-substitution and actual Step-2-infilled motion. For a quick decision we retain the DP oracle, fixed 3/7/15 controls, adaptive placement-only, adaptive deployable, and fixed-7 deployable conditions.

In [ ]:
MOTION_CONDITIONS = [
    'step2_dp_oracle_gt_anchors',
    'fixed_gap3_gt_anchors',
    'fixed_gap7_gt_anchors',
    'fixed_gap15_gt_anchors',
    f'{PRIMARY_LABEL}__adaptive_gt_history',
    f'{PRIMARY_LABEL}__adaptive_generated_history',
    f'{PRIMARY_LABEL}__fixed_gap7_generated_history',
]
motion_command = [
    str(PYTHON_EXECUTABLE),
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_adaptive_motion.py'),
    '--adaptive_output_dir', str(ADAPTIVE_OUTPUT),
    '--output_dir', str(MOTION_OUTPUT),
    '--step2_config', str(STEP2_CONFIG),
    '--step2_checkpoint', str(STEP2_CHECKPOINT),
    '--device', DEVICE,
    '--step2_batch_size', str(STEP2_BATCH_SIZE),
    '--fid_batch_size', '64',
    '--metric_seed', str(SUBSET_SEED),
    '--export_only',
]
for condition in MOTION_CONDITIONS:
    motion_command.extend(['--condition', condition])
print(' '.join(motion_command))
if RUN_MOTION_EXPORT:
    subprocess.run(motion_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Skipped; reading existing exports.')


In [ ]:
# FID is separated so an interrupted evaluator run does not repeat Step 2 inference.
fid_command = [
    str(PYTHON_EXECUTABLE),
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_adaptive_motion.py'),
    '--adaptive_output_dir', str(ADAPTIVE_OUTPUT),
    '--output_dir', str(MOTION_OUTPUT),
    '--device', DEVICE,
    '--fid_batch_size', '64',
    '--metric_seed', str(SUBSET_SEED),
    '--fid_only',
]
print(' '.join(fid_command))
if RUN_FID:
    subprocess.run(fid_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Skipped; reading existing FID outputs.')


In [ ]:
decoded = pd.read_csv(MOTION_OUTPUT / 'decoded_metrics_summary.csv')
fid = pd.read_csv(MOTION_OUTPUT / 'adaptive_motion_fid.csv')

# Codec-relative reference removes the codec floor and is the cleanest comparison
# among sparse scheduling conditions. Raw-motion results remain in the CSV/report.
codec_fid = fid[fid['reference'] == 'causal_codec_reconstruction'].copy()
display(decoded.sort_values(['protocol', 'codec_relative_rmse']))
display(codec_fid.sort_values(['protocol', 'FID_norm_by_GT']))


## Interpretation order

Use the results in this order:

1. **Rate:** confirm the adaptive mean gap/anchor fraction is close enough to fixed gap 7 for a fair comparison.
2. **Placement:** compare `adaptive_gt_history` with `fixed_gap7_gt_anchors` in the **Step-2-infilled** rows. If adaptive loses here, the scheduler itself is not helping.
3. **Anchor content:** compare adaptive anchor-substitution FID with fixed-7 generated anchor-substitution FID. This isolates anchor prediction from infilling.
4. **Deployable system:** compare adaptive generated-history Step-2 FID with fixed-7 generated-history Step-2 FID.
5. **Headroom:** compare adaptive placement-only with the DP oracle. A large gap means schedule imitation remains weak.

Proceed to self-forcing fine-tuning only if placement is useful but generated-history performance is substantially worse. If adaptive placement itself cannot beat the rate-matched fixed control, exposure-bias fine-tuning is not the immediate problem.